# Popularity Bias Analysis

One of the most well-documented problems in recommender systems is **popularity bias**:
algorithms tend to recommend popular items far more often than unpopular ones,
even when personalised recommendations should theoretically surface niche content.

This notebook quantifies how severe this bias is across our seven algorithms,
and explains why it matters for the MovieLens dataset specifically.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Source modules
from src.data_loading import load_ratings, load_items, train_test_split_ratings
from src.baselines import MostPopularRecommender, HighestAverageRatingRecommender, RandomRecommender
from src.content_based import ContentBasedRecommender
from src.collaborative_filtering import ItemItemCollaborativeFiltering, UserUserCollaborativeFiltering
from src.matrix_factorization import MatrixFactorizationRecommender
from src import config

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA = Path('../data/raw')
K    = 10
N_EVAL_USERS = 100   # fewer for speed in notebook

## 1. Load data and train models

In [ ]:
ratings = load_ratings(DATA / 'ratings.csv')
movies  = load_items(DATA / 'movies.csv')
tags    = pd.read_csv(DATA / 'tags.csv')

train, test = train_test_split_ratings(ratings)
print(f'Train: {len(train):,} | Test: {len(test):,}')

# Global popularity (by number of ratings in the full dataset)
popularity = ratings.groupby('movieId')['rating'].count()
n_users = ratings['userId'].nunique()

# Pre-group training set by user for O(1) lookup in recommendation loops
train_by_user = {uid: grp for uid, grp in train.groupby('userId')}

In [ ]:
# Train all models (this will take ~2-3 minutes)
models = {}

models['Most Popular']    = MostPopularRecommender()
models['Highest Avg']     = HighestAverageRatingRecommender()
models['Random']          = RandomRecommender()
models['Content-Based']   = ContentBasedRecommender(use_tags=False)
models['CB + Tags']       = ContentBasedRecommender(use_tags=True)
models['Item-Item CF']    = ItemItemCollaborativeFiltering(k=20)
models['User-User CF']    = UserUserCollaborativeFiltering(k=20)
models['Matrix Fact']     = MatrixFactorizationRecommender(n_factors=50, n_epochs=20)

for name, model in models.items():
    if 'Content' in name or 'CB' in name:
        model.fit(train, movies, tags)
    elif name in ('Item-Item CF', 'User-User CF', 'Matrix Fact'):
        model.fit(train)
    else:
        model.fit(train, movies)
    print(f'  {name} trained')

# Show MF convergence curve
mf_losses = models['Matrix Fact'].train_losses_
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(mf_losses)+1), mf_losses, color='#e94560', marker='o', markersize=3)
plt.xlabel('Epoch'); plt.ylabel('Train RMSE')
plt.title('Matrix Factorisation , training loss per epoch')
plt.tight_layout()
plt.savefig('../results/figures/mf_convergence.png', bbox_inches='tight')
plt.show()
print(f'MF converged: RMSE {mf_losses[0]:.3f} → {mf_losses[-1]:.3f}')

## 2. Popularity tiers: Head, Torso, Tail

I divide the catalog into three tiers based on how many ratings each movie received:
- **Head** (top 20% of ratings) , the blockbusters
- **Torso** (middle 30%) , well-known but not mega-popular
- **Tail** (bottom 50%) , niche, obscure, or very new films

Note: I'm defining tiers by rating *counts*, not by movie count.
Because of the long tail, the tail contains the vast majority of movies.

In [ ]:
pop_df = popularity.reset_index()
pop_df.columns = ['movieId', 'n_ratings']
pop_df = pop_df.sort_values('n_ratings', ascending=False).reset_index(drop=True)

cumulative = pop_df['n_ratings'].cumsum() / pop_df['n_ratings'].sum()
head_cut   = (cumulative <= 0.20).sum()
torso_cut  = (cumulative <= 0.50).sum()

def assign_tier(row_idx):
    if row_idx < head_cut:
        return 'Head'
    elif row_idx < torso_cut:
        return 'Torso'
    else:
        return 'Tail'

pop_df['tier']  = [assign_tier(i) for i in range(len(pop_df))]
item_to_tier    = dict(zip(pop_df['movieId'], pop_df['tier']))

tier_counts = pop_df['tier'].value_counts()
print('Movies per tier:')
for tier in ['Head', 'Torso', 'Tail']:
    n = tier_counts.get(tier, 0)
    print(f'  {tier:6s}: {n:,} movies ({n/len(pop_df):.1%})')

TIER_COLORS = {'Head': '#e94560', 'Torso': '#f0a500', 'Tail': '#4361ee'}

fig, ax = plt.subplots(figsize=(13, 4))
for tier, color in TIER_COLORS.items():
    mask = pop_df['tier'] == tier
    ax.fill_between(pop_df.index[mask], pop_df['n_ratings'][mask],
                    alpha=0.7, color=color, label=tier)

ax.set_xlabel('Movie rank (by popularity)')
ax.set_ylabel('Number of ratings')
ax.set_title('Popularity tiers , Head / Torso / Tail')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/popularity_tiers.png', bbox_inches='tight')
plt.show()

## 3. Collect recommendations for a sample of users

In [ ]:
all_users  = train['userId'].unique()
rng        = np.random.default_rng(42)
eval_users = rng.choice(all_users, size=min(N_EVAL_USERS, len(all_users)), replace=False)

# For each model, collect all recommendations
# Note: recommend() expects a ratings DataFrame for the user, not a set of seen items
all_recs = {}

for name, model in models.items():
    recs = []
    for uid in eval_users:
        user_train_df = train_by_user.get(uid)
        if user_train_df is None:
            continue
        try:
            r = model.recommend(uid, user_train_df, n=K)
            recs.extend(r)
        except Exception:
            pass
    all_recs[name] = recs
    print(f'  {name}: {len(recs)} total recommendations')

## 4. Tier distribution per algorithm

For each model, what fraction of its recommendations come from each popularity tier?

In [ ]:
tier_fracs = {}
for name, recs in all_recs.items():
    tiers = [item_to_tier.get(mid, 'Tail') for mid in recs]
    total = len(tiers) or 1
    tier_fracs[name] = {
        'Head':  tiers.count('Head')  / total,
        'Torso': tiers.count('Torso') / total,
        'Tail':  tiers.count('Tail')  / total,
    }

frac_df = pd.DataFrame(tier_fracs).T

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(frac_df))
for tier, color in TIER_COLORS.items():
    vals = frac_df[tier].values
    bars = ax.bar(frac_df.index, vals, bottom=bottom, color=color,
                  label=tier, width=0.6)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    f'{v:.0%}', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottom += vals

ax.set_ylabel('Fraction of recommendations')
ax.set_title('Popularity tier distribution per algorithm')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(loc='upper right')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/tier_distribution.png', bbox_inches='tight')
plt.show()

print(frac_df.round(3).to_string())

## 5. Average popularity of recommendations

A simpler view: what is the mean number of ratings for items each algorithm recommends?
Higher = more biased toward popular items.

In [ ]:
avg_pop = {}
for name, recs in all_recs.items():
    pops = [popularity.get(mid, 0) for mid in recs]
    avg_pop[name] = np.mean(pops) if pops else 0

avg_pop_s = pd.Series(avg_pop).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e94560' if v > avg_pop_s.median() else '#4361ee' for v in avg_pop_s.values]
ax.barh(avg_pop_s.index, avg_pop_s.values, color=colors)
ax.axvline(popularity.mean(), color='white', linestyle='--', alpha=0.5,
           label=f'Catalog mean ({popularity.mean():.0f} ratings/movie)')
ax.set_xlabel('Mean ratings of recommended items')
ax.set_title('Average popularity of recommendations per algorithm')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/avg_popularity.png', bbox_inches='tight')
plt.show()

print('Catalog mean popularity:', f'{popularity.mean():.1f}')
for name, val in avg_pop_s.items():
    ratio = val / popularity.mean()
    print(f'  {name:20s}: {val:6.1f} ratings/movie  ({ratio:.1f}x catalog mean)')

## 6. Gini coefficient: inequality of recommendations

The **Gini coefficient** measures inequality. Here I compute it over the *recommendation frequency*
of each movie: how unequally are recommendations distributed across the catalog?

- Gini = 0 → every movie recommended equally often (perfectly fair)
- Gini = 1 → one movie gets all recommendations (perfectly unfair)

A good recommender should have a lower Gini than a purely popularity-based one,
because it surfaces diverse long-tail content.

In [ ]:
def gini(values):
    arr = np.sort(np.array(values, dtype=float))
    n   = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return (2 * (idx * arr).sum()) / (n * arr.sum()) - (n + 1) / n

all_items_arr = movies['movieId'].values
gini_scores   = {}

for name, recs in all_recs.items():
    freq = pd.Series(recs).value_counts().reindex(all_items_arr, fill_value=0)
    gini_scores[name] = gini(freq.values)

gini_s = pd.Series(gini_scores).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(gini_s.index, gini_s.values, color='#06d6a0')
ax.set_xlabel('Gini coefficient (lower = more diverse)')
ax.set_title('Recommendation inequality (Gini) per algorithm')
ax.set_xlim(0, 1)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/gini.png', bbox_inches='tight')
plt.show()

for name, g in gini_s.items():
    print(f'  {name:20s}: Gini = {g:.4f}')

## 7. Novelty score

**Novelty** measures how surprising recommendations are to the average user.
Using self-information: novelty(i) = -log₂(pop(i) / n_users).
A rarely-seen movie has high self-information (high novelty);
a movie everyone has seen has low self-information (low novelty).

In [ ]:
def novelty(recs, popularity, n_users, k=10):
    scores = []
    for mid in recs[:k]:
        p = popularity.get(mid, 1) / n_users
        scores.append(-np.log2(p + 1e-10))
    return np.mean(scores) if scores else 0

novelty_per_user = {name: [] for name in models}

for name, model in models.items():
    for uid in eval_users:
        user_train_df = train_by_user.get(uid)
        if user_train_df is None:
            continue
        try:
            recs = model.recommend(uid, user_train_df, n=K)
            novelty_per_user[name].append(novelty(recs, popularity, n_users, K))
        except Exception:
            pass

novelty_means = {k: np.mean(v) for k, v in novelty_per_user.items()}
novelty_s     = pd.Series(novelty_means).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(novelty_s.index, novelty_s.values, color='#f0a500')
ax.set_xlabel('Mean novelty score (higher = more novel)')
ax.set_title('Novelty@10 per algorithm')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/novelty.png', bbox_inches='tight')
plt.show()

## 8. Summary and interpretation

### What we found

| Algorithm | Bias direction | Why |
|---|---|---|
| Most Popular | Maximum head bias | Literally recommends the most-rated movies |
| Highest Avg | Moderate head bias | Min-rating filter concentrates on well-known films |
| Item-Item CF | High head bias | Similarity dominated by co-occurrence → popular items have more co-occurrences |
| User-User CF | Moderate head bias | Similar users also gravitate toward popular films |
| Matrix Fact | Moderate head bias | Latent factors capture general popularity signal |
| Content-Based | Lower head bias | Operates on genres, not interaction data , less susceptible to popularity |
| Random | Unbiased | Uniform sample, no feedback loop |

### Why this matters

Popularity bias creates a **feedback loop** in production systems:
1. Popular items get recommended → they get more views
2. More views → more ratings → more data → perceived as even more popular
3. Long-tail content is systematically starved of exposure

This is bad for:
- **Users** , they miss out on niche content they would actually enjoy
- **Catalog owners** , most of their inventory is never surfaced
- **Fairness** , items / creators with early traction dominate permanently

### Mitigation strategies (not implemented here, but worth knowing)

- **Inverse Propensity Scoring** , weight rare interactions more heavily in training
- **Regularisation on item exposure** , penalise concentration on few items
- **Re-ranking** , post-filter recommendations to enforce minimum tail representation
- **Separate popularity debiasing layer** , subtract item popularity signal from scores